In [ ]:
import sys
import os
import json
import re
import pandas as pd
import random
from tqdm import tqdm
from qdrant_client import QdrantClient, models
from sentence_transformers import CrossEncoder
from langchain_ollama import OllamaLLM
from typing import TypedDict, List
from langgraph.graph import StateGraph, START, END

import transformers.utils.import_utils
def dummy_check(): return False
transformers.utils.import_utils.is_torch_fx_available = dummy_check
sys.modules['transformers.utils.import_utils'].is_torch_fx_available = dummy_check

from FlagEmbedding import BGEM3FlagModel

client = QdrantClient(host="127.0.0.1", port=6333)
COLLECTION_NAME = "legal_hybrid_base"

model = BGEM3FlagModel('BAAI/bge-m3', use_fp16=False, device='cpu')
reranker = CrossEncoder('BAAI/bge-reranker-v2-m3', max_length=512, device='cpu')

llm = OllamaLLM(
    model="qwen2.5:7b", 
    temperature=0,
    num_ctx=2048 
)

Fetching 30 files:   0%|          | 0/30 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

In [ ]:
client.create_payload_index(
    collection_name=COLLECTION_NAME,
    field_name="text",
    field_schema=models.TextIndexParams(
        type="text",
        tokenizer="word",
        min_token_len=1,
        lowercase=True
    )
)

def ultimate_hybrid_search(query_text, fetch_limit=10, final_limit=3):
    print(f"\nИщем кандидатов для: '{query_text}'")

    output = model.encode(query_text, return_dense=True, return_sparse=True, return_colbert_vecs=False)
    dense_vec = output['dense_vecs'].tolist()
    
    unique_tokens = {}
    for token_str, weight in output['lexical_weights'].items():
        token_id = model.tokenizer.convert_tokens_to_ids(token_str)
        if token_id is not None:
            unique_tokens[token_id] = max(unique_tokens.get(token_id, 0), float(weight))
            
    query_sparse = models.SparseVector(
        indices=list(unique_tokens.keys()), 
        values=list(unique_tokens.values())
    )

    exact_terms = re.findall(r'\b\d+(?:-\d+)*\b|[А-Яа-яA-Za-z]+-\d+(?:-\d+)*', query_text)
    search_filter = None
    if exact_terms:
        must_conditions = [
            models.FieldCondition(key="text", match=models.MatchText(text=term)) 
            for term in exact_terms
        ]
        search_filter = models.Filter(must=must_conditions)

    prefetches = [
        models.Prefetch(query=dense_vec, using="dense", limit=fetch_limit, filter=search_filter),
        models.Prefetch(query=query_sparse, using="sparse", limit=fetch_limit, filter=search_filter),
    ]
    
    results = client.query_points(
        collection_name=COLLECTION_NAME,
        prefetch=prefetches,
        query=models.FusionQuery(fusion=models.Fusion.RRF),
        limit=fetch_limit,
        with_payload=True
    ).points

    if not results and search_filter:
        print("Не нашел точного совпадения. Отключаем фильтр и ищем по смыслу")
        prefetches_fallback = [
            models.Prefetch(query=dense_vec, using="dense", limit=fetch_limit),
            models.Prefetch(query=query_sparse, using="sparse", limit=fetch_limit),
        ]
        results = client.query_points(
            collection_name=COLLECTION_NAME,
            prefetch=prefetches_fallback,
            query=models.FusionQuery(fusion=models.Fusion.RRF),
            limit=fetch_limit,
            with_payload=True
        ).points

    if not results:
        return []

    print(f"Кандидатов от Qdrant: {len(results)}. Запускаем реранкер...")
    
    pairs = [[query_text, chunk.payload['text']] for chunk in results]
    rerank_scores = reranker.predict(pairs)
    
    if hasattr(rerank_scores, 'tolist'):
        rerank_scores = rerank_scores.tolist()
    if not isinstance(rerank_scores, list):
        rerank_scores = [rerank_scores]
        
    reranked_results = sorted(zip(rerank_scores, results), key=lambda x: x[0], reverse=True)
    
    top_results = []
    for rank, (score, chunk) in enumerate(reranked_results[:final_limit]):
        chunk.score = score
        top_results.append(chunk)
        
    return top_results

In [ ]:
class GraphState(TypedDict):
    question: str       
    documents: List[str] 
    generation: str      

def retrieve_node(state: GraphState):
    question = state["question"]
    results = ultimate_hybrid_search(question, final_limit=3)
    documents = []
    for i, chunk in enumerate(results):
        text = chunk.payload.get('text', '')
        source = chunk.payload.get('source', f'Фрагмент №{i+1}') 
        documents.append(f"--- ИСТОЧНИК: {source} ---\n{text}")
    if not documents:
        documents = ["К сожалению, в базе не найдено релевантных статей закона."]
    return {"documents": documents}

def generate_node(state: GraphState):
    question = state["question"]
    documents = state["documents"]
    context = "\n\n".join(documents)
    prompt = f"""Ты - профессиональный юрист-консультант. 
Твоя задача - ответить на вопрос пользователя, используя ТОЛЬКО предоставленный контекст из законов РФ.
Если в контексте нет ответа на вопрос, честно скажи: "Я не могу ответить на этот вопрос на основе предоставленных законов".
Не придумывай информацию от себя.

ВАЖНОЕ ПРАВИЛО: В конце ответа обязательно напиши заголовок "Источники:" и перечисли названия документов из контекста, на которые ты опирался.

Контекст (выдержки из законов):
{context}

Вопрос пользователя: {question}

Ответ:"""
    response = llm.invoke(prompt)
    return {"generation": response}

workflow = StateGraph(GraphState)
workflow.add_node("retrieve", retrieve_node)
workflow.add_node("generate", generate_node)
workflow.add_edge(START, "retrieve")
workflow.add_edge("retrieve", "generate")
workflow.add_edge("generate", END)
app = workflow.compile()

In [ ]:
eval_dataset = [
    {
        "question": "Какой максимальный срок может длиться проверка организации в обычных условиях?",
        "expected_source": "Закон Российской Федерации от 21.07.1993 №5485-1.docx"
    },
    {
        "question": "Какую информацию, касающуюся экологии и здравоохранения, закон запрещает засекречивать?",
        "expected_source": "Закон Российской Федерации от 21.07.1993 №5485-1.docx"
    },
    {
        "question": "Что является основанием для прекращения допуска сотрудника к гостайне по статье 23?",
        "expected_source": "Закон Российской Федерации от 21.07.1993 №5485-1.docx"
    },
    {
        "question": "Куда может обратиться иностранная некоммерческая организация для обжалования действий госорганов?",
        "expected_source": "Федеральный закон от 12.01.1996 №7-ФЗ" 
    },
    {
        "question": "Как осуществляется мониторинг в сфере отношений с соотечественниками за рубежом согласно статье 25?",
        "expected_source": "Федеральный закон от 24.05.1999 №99-ФЗ"
    }
]

successful_hits = 0
mrr_score = 0.0
total_questions = len(eval_dataset)

for idx, item in enumerate(eval_dataset):
    question = item["question"]
    expected_doc = item["expected_source"]
    print(f"[{idx+1}/{total_questions}] Вопрос: {question}")
    results = ultimate_hybrid_search(question, final_limit=3)
    hit = False
    for rank, chunk in enumerate(results):
        if expected_doc in chunk.payload['source']:
            hit = True
            successful_hits += 1
            mrr_score += 1.0 / (rank + 1)
            print(f" Успех. Правильный документ найден на {rank+1} месте")
            break 
    if not hit:
        print(f" Провал. Правильный документ не попал в Топ-3")

hit_rate = (successful_hits / total_questions) * 100
mrr_final = mrr_score / total_questions
print(f"\nФИНАЛЬНЫЕ МЕТРИКИ:\nHit Rate: {hit_rate:.1f}%\nMRR : {mrr_final:.3f}")

[1/5] Вопрос: Какой максимальный срок может длиться проверка организации в обычных условиях?

Ищем кандидатов для: 'Какой максимальный срок может длиться проверка организации в обычных условиях?'
Кандидатов от Qdrant: 10. Запускаем реранкер...
 Успех. Правильный документ найден на 1 месте
[2/5] Вопрос: Какую информацию, касающуюся экологии и здравоохранения, закон запрещает засекречивать?

Ищем кандидатов для: 'Какую информацию, касающуюся экологии и здравоохранения, закон запрещает засекречивать?'
Кандидатов от Qdrant: 10. Запускаем реранкер...
 Успех. Правильный документ найден на 1 месте
[3/5] Вопрос: Что является основанием для прекращения допуска сотрудника к гостайне по статье 23?

Ищем кандидатов для: 'Что является основанием для прекращения допуска сотрудника к гостайне по статье 23?'
Кандидатов от Qdrant: 10. Запускаем реранкер...
 Успех. Правильный документ найден на 1 месте
[4/5] Вопрос: Куда может обратиться иностранная некоммерческая организация для обжалования действий го

In [ ]:
gt_list =[
  {
    "question": "Каковы основные цели функционирования Системы мониторинга в сфере межнациональных отношений?",
    "context": "Целью создания и функционирования Системы мониторинга является реализация Стратегии государственной национальной политики...",
    "ground_truth": "Система мониторинга создана для реализации Стратегии государственной национальной политики РФ...",
    "source": "Государственная информационная система мониторинга..."
  },
  {
    "question": "Какие сведения закон запрещает относить к государственной тайне и засекречивать?",
    "context": "Не подлежат отнесению к государственной тайне и засекречиванию сведения: о чрезвычайных происшествиях и катастрофах...",
    "ground_truth": "К государственной тайне нельзя относить сведения о ЧП и катастрофах, угрожающих здоровью граждан...",
    "source": "Закон РФ от 21.07.1993 №5485-1"
  },
  {
    "question": "Каков максимальный срок засекречивания сведений, составляющих гостайну?",
    "context": "Срок засекречивания сведений, составляющих государственную тайну, не должен превышать 30 лет.",
    "ground_truth": "По общему правилу срок засекречивания не может быть более 30 лет.",
    "source": "Закон РФ от 21.07.1993 №5485-1"
  },
  {
    "question": "Какая организация по закону признается некоммерческой?",
    "context": "Некоммерческой организацией является организация, не имеющая извлечение прибыли в качестве основной цели...",
    "ground_truth": "Некоммерческой организацией (НКО) считается такая организация, основной целью которой не является получение прибыли.",
    "source": "Федеральный закон от 12.01.1996 №7-ФЗ"
  },
  {
    "question": "Что признается крупной сделкой для автономной некоммерческой организации?",
    "context": "Для целей настоящего Федерального закона крупной сделкой автономной некоммерческой организации признается сделка...",
    "ground_truth": "Крупной сделкой для АНО считается сделка (или несколько взаимосвязанных сделок) по распоряжению деньгами или имуществом...",
    "source": "Федеральный закон от 12.01.1996 №7-ФЗ"
  },
  {
    "question": "Как часто должны проводиться заседания Консультативного совета по делам национально-культурных автономий?",
    "context": "Заседания Совета проводятся не реже двух раз в год.",
    "ground_truth": "Заседания Консультативного совета должны проходить как минимум дважды в год.",
    "source": "Приказ ФАДН России от 23.10.2015 №70"
  },
  {
    "question": "Кто именно признается соотечественником согласно российскому законодательству?",
    "context": "Соотечественниками являются лица, родившиеся в одном государстве, проживающие либо проживавшие в нем...",
    "ground_truth": "Соотечественники — это лица, имеющие общие корни, язык, культуру и историю, а также их прямые потомки.",
    "source": "Федеральный закон от 24.05.1999 №99-ФЗ"
  },
  {
    "question": "Какие права имеют соотечественники в области образования в Российской Федерации?",
    "context": "Органы государственной власти... способствуют получению соотечественниками образования...",
    "ground_truth": "Соотечественники, не имеющие гражданства РФ, имеют право на доступ к образованию наравне с россиянами...",
    "source": "Федеральный закон от 24.05.1999 №99-ФЗ"
  },
  {
    "question": "В каких формах могут создаваться общины коренных малочисленных народов?",
    "context": "Общинами коренных малочисленных народов Российской Федерации... признаются формы самоорганизации лиц...",
    "ground_truth": "Общины малочисленных народов могут создаваться как формы самоорганизации, основанные на кровнородственном...",
    "source": "Федеральный закон от 12.01.1996 №7-ФЗ"
  },
  {
    "question": "Как иностранная НПО может осуществлять деятельность на территории России?",
    "context": "Иностранная некоммерческая неправительственная организация осуществляет свою деятельность на территории Российской Федерации...",
    "ground_truth": "Для ведения деятельности в РФ иностранная некоммерческая неправительственная организация должна создать структурные подразделения...",
    "source": "Федеральный закон от 12.01.1996 №7-ФЗ"
  }
]

with open('ground_truth_dataset.json', 'w', encoding='utf-8') as f:
    json.dump(gt_list, f, ensure_ascii=False, indent=4)

In [9]:
from datasets import Dataset
from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy, answer_correctness
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas.run_config import RunConfig
from langchain_huggingface import HuggingFaceEmbeddings

model_name = "BAAI/bge-m3"
model_kwargs = {'device': 'cpu'} 
encode_kwargs = {'normalize_embeddings': True}

embeddings = HuggingFaceEmbeddings(
    model_name=model_name,
    model_kwargs=model_kwargs,
    encode_kwargs=encode_kwargs
)

with open('ground_truth_dataset.json', 'r', encoding='utf-8') as f:
    gt_data = json.load(f)

results = []
for entry in tqdm(gt_data):
    question = entry['question']
    try:
        response = app.invoke({"question": question})
        results.append({
            "question": question,
            "answer": response.get('generation', ""),
            "contexts": response.get('documents', []),
            "ground_truth": entry.get('ground_truth', "")
        })
    except Exception as e:
        print(f"Ошибка: {e}")

dataset = Dataset.from_list(results)
ragas_llm = LangchainLLMWrapper(llm)    
ragas_embeddings = LangchainEmbeddingsWrapper(embeddings)
run_config = RunConfig(max_workers=1 , timeout=1600)

score = evaluate(
    dataset=dataset,
    metrics=[faithfulness, answer_relevancy, answer_correctness],
    llm=ragas_llm,
    embeddings=ragas_embeddings,
    run_config=run_config,
    raise_exceptions=False
)

print(score)
df_results = score.to_pandas()
df_results.to_csv("rag_evaluation_final_report.csv", index=False, encoding='utf-8-sig')

C:\Users\asus\AppData\Local\Temp\ipykernel_34648\3394586275.py:3: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import faithfulness, answer_relevancy, answer_correctness
C:\Users\asus\AppData\Local\Temp\ipykernel_34648\3394586275.py:3: DeprecationWarning: Importing answer_relevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import answer_relevancy
  from ragas.metrics import faithfulness, answer_relevancy, answer_correctness
C:\Users\asus\AppData\Local\Temp\ipykernel_34648\3394586275.py:3: DeprecationWarning: Importing answer_correctness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections impor

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

  0%|          | 0/10 [00:00<?, ?it/s]


Ищем кандидатов для: 'Каковы основные цели функционирования Системы мониторинга в сфере межнациональных отношений?'
Кандидатов от Qdrant: 10. Запускаем реранкер...


 10%|█         | 1/10 [00:47<07:04, 47.18s/it]

...


Ищем кандидатов для: 'Как иностранная НПО может осуществлять деятельность на территории России?'
Кандидатов от Qdrant: 10. Запускаем реранкер...


100%|██████████| 10/10 [05:34<00:00, 33.48s/it]
C:\Users\asus\AppData\Local\Temp\ipykernel_34648\3394586275.py:37: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  ragas_llm = LangchainLLMWrapper(llm)
C:\Users\asus\AppData\Local\Temp\ipykernel_34648\3394586275.py:38: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  ragas_embeddings = LangchainEmbeddingsWrapper(embeddings)


Evaluating:   0%|          | 0/30 [00:00<?, ?it/s]

Prompt fix_output_format failed to parse output: The output parser failed to parse the output including retries.
Prompt fix_output_format failed to parse output: The output parser failed to parse the output including retries.
Prompt fix_output_format failed to parse output: The output parser failed to parse the output including retries.
Prompt n_l_i_statement_prompt failed to parse output: The output parser failed to parse the output including retries.
Exception raised in Job[0]: RagasOutputParserException(The output parser failed to parse the output including retries.)
Prompt fix_output_format failed to parse output: The output parser failed to parse the output including retries.
Prompt fix_output_format failed to parse output: The output parser failed to parse the output including retries.
Prompt fix_output_format failed to parse output: The output parser failed to parse the output including retries.
Prompt n_l_i_statement_prompt failed to parse output: The output parser failed to pa

{'faithfulness': 1.0000, 'answer_relevancy': 0.6895, 'answer_correctness': 0.3557}
